# OpenPlaque Validation Notebook

This notebook validates the pretrained `Dataset001_CCTA_DHM` nnU-Net plaque model against the authors' sample dataset.

It starts from a fresh Colab runtime and performs:

1. Mount Google Drive
2. Clone/update OpenPlaque from GitHub
3. Install requirements
4. Configure nnU-Net
5. Extract model weights if needed
6. Download sample dataset if needed
7. Run inference on all sample cases
8. Compare predictions to ground-truth labels
9. Compute Dice, IoU, precision, recall, and volume error
10. Save a CSV validation report to Google Drive

Expected Google Drive model ZIP:

```text
/content/drive/MyDrive/OpenPlaque/models/Dataset001_CCTA_DHM-20260703T233210Z-3-001.zip
```

Research use only. Not for clinical decision-making.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
# Clone or update OpenPlaque
!git clone https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque || true
!git -C /content/OpenPlaque pull


In [ ]:
# Install requirements
!wget -q https://raw.githubusercontent.com/pazzani/OpenPlaque/main/requirements-colab.txt -O /content/requirements-colab.txt
!pip install -q -r /content/requirements-colab.txt
!pip install -q gdown


In [ ]:
# Configure Python path and nnU-Net environment
import os
import sys
from pathlib import Path

repo = Path("/content/OpenPlaque")
src = repo / "src"

if str(src) not in sys.path:
    sys.path.insert(0, str(src))

os.environ["nnUNet_raw"] = "/content/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = "/content/nnUNet_preprocessed"
os.environ["nnUNet_results"] = "/content/nnUNet_results"

for d in [os.environ["nnUNet_raw"], os.environ["nnUNet_preprocessed"], os.environ["nnUNet_results"]]:
    os.makedirs(d, exist_ok=True)

print("OpenPlaque src:", src)
print("nnUNet_results:", os.environ["nnUNet_results"])


In [ ]:
# Extract nnU-Net model weights if needed
import zipfile
from pathlib import Path

model_zip = Path("/content/drive/MyDrive/OpenPlaque/models/Dataset001_CCTA_DHM-20260703T233210Z-3-001.zip")
model_target = Path("/content/nnUNet_results/Dataset001_CCTA_DHM")

if model_target.exists():
    print("Model already extracted:", model_target)
else:
    if not model_zip.exists():
        raise FileNotFoundError(f"Missing model ZIP: {model_zip}")
    print("Extracting model ZIP...")
    with zipfile.ZipFile(model_zip) as z:
        z.extractall("/content/nnUNet_results")
    print("Extracted model to /content/nnUNet_results")

!find /content/nnUNet_results -maxdepth 3


## Download the authors' sample dataset

This is the Google Drive folder referenced in the model README.


In [ ]:
# Download sample dataset if needed
from pathlib import Path

sample_root = Path("/content/sample_dataset/Sample_Dataset")
images_dir = sample_root / "Images"
labels_dir = sample_root / "Labels"

if images_dir.exists() and labels_dir.exists():
    print("Sample dataset already exists.")
else:
    !rm -rf /content/sample_dataset
    !mkdir -p /content/sample_dataset
    !gdown --folder "https://drive.google.com/drive/folders/1i4XD-GfP-wS50m9smGR1LzBRJokKro9_?usp=sharing" -O /content/sample_dataset --remaining-ok

print("Images:", len(list(images_dir.glob("*.nii.gz"))))
print("Labels:", len(list(labels_dir.glob("*.nii.gz"))))


In [ ]:
# Inspect sample filenames
!find /content/sample_dataset/Sample_Dataset -maxdepth 2 | head -80


## Validation utilities

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import SimpleITK as sitk
import matplotlib.pyplot as plt


def dice_score(pred, gt, label=2):
    p = pred == label
    g = gt == label
    denom = p.sum() + g.sum()
    if denom == 0:
        return np.nan
    return 2 * np.logical_and(p, g).sum() / denom


def iou_score(pred, gt, label=2):
    p = pred == label
    g = gt == label
    union = np.logical_or(p, g).sum()
    if union == 0:
        return np.nan
    return np.logical_and(p, g).sum() / union


def binary_metrics(pred, gt, label=2):
    p = pred == label
    g = gt == label

    tp = int(np.logical_and(p, g).sum())
    fp = int(np.logical_and(p, ~g).sum())
    fn = int(np.logical_and(~p, g).sum())
    tn = int(np.logical_and(~p, ~g).sum())

    precision = tp / (tp + fp) if (tp + fp) else np.nan
    recall = tp / (tp + fn) if (tp + fn) else np.nan
    specificity = tn / (tn + fp) if (tn + fp) else np.nan

    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "dice": dice_score(pred, gt, label),
        "iou": iou_score(pred, gt, label),
    }


def run_prediction(image_path, out_root="/content/validation_predictions"):
    image_path = Path(image_path)
    case_id = image_path.name.replace("_0000.nii.gz", "")

    input_dir = Path("/content/validation_input") / case_id
    output_dir = Path(out_root)

    shutil.rmtree(input_dir, ignore_errors=True)
    input_dir.mkdir(parents=True, exist_ok=True)
    output_dir.mkdir(parents=True, exist_ok=True)

    shutil.copy(image_path, input_dir / f"{case_id}_0000.nii.gz")

    cmd = [
        "nnUNetv2_predict",
        "-i", str(input_dir),
        "-o", str(output_dir),
        "-d", "Dataset001_CCTA_DHM",
        "-c", "3d_fullres",
        "-f", "0", "1", "2", "3", "4",
    ]

    subprocess.run(cmd, check=True)

    pred_path = output_dir / f"{case_id}.nii.gz"
    return case_id, pred_path


def load_array(path):
    img = sitk.ReadImage(str(path))
    arr = sitk.GetArrayFromImage(img)
    return img, arr


def label_path_for_image(image_path):
    image_path = Path(image_path)
    base = image_path.name.replace("_0000.nii.gz", ".nii.gz")
    return labels_dir / base


## Run validation on all sample cases

This may take a while because it runs five-fold nnU-Net inference for every sample artery segment.


In [ ]:
image_files = sorted(images_dir.glob("*.nii.gz"))
print("Cases:", len(image_files))

results = []

for i, image_path in enumerate(image_files, start=1):
    print(f"\n[{i}/{len(image_files)}] {image_path.name}")

    label_path = label_path_for_image(image_path)
    if not label_path.exists():
        print("  Missing label:", label_path)
        continue

    case_id, pred_path = run_prediction(image_path)

    pred_img, pred = load_array(pred_path)
    gt_img, gt = load_array(label_path)

    if pred.shape != gt.shape:
        print("  Shape mismatch:", pred.shape, gt.shape)
        continue

    spacing = pred_img.GetSpacing()
    voxel_volume = float(np.prod(spacing))

    metrics = binary_metrics(pred, gt, label=2)

    pred_voxels = int((pred == 2).sum())
    gt_voxels = int((gt == 2).sum())

    row = {
        "case_id": case_id,
        "image": str(image_path),
        "label": str(label_path),
        "prediction": str(pred_path),
        "spacing_x": spacing[0],
        "spacing_y": spacing[1],
        "spacing_z": spacing[2],
        "voxel_volume_mm3": voxel_volume,
        "gt_plaque_voxels": gt_voxels,
        "pred_plaque_voxels": pred_voxels,
        "gt_plaque_volume_mm3": gt_voxels * voxel_volume,
        "pred_plaque_volume_mm3": pred_voxels * voxel_volume,
        "volume_error_mm3": (pred_voxels - gt_voxels) * voxel_volume,
        **metrics,
    }

    results.append(row)

df = pd.DataFrame(results)
df.head()


## Summary metrics

In [ ]:
summary = {
    "n_cases": len(df),
    "mean_dice": df["dice"].mean(),
    "median_dice": df["dice"].median(),
    "mean_iou": df["iou"].mean(),
    "median_iou": df["iou"].median(),
    "mean_precision": df["precision"].mean(),
    "mean_recall": df["recall"].mean(),
    "mean_abs_volume_error_mm3": df["volume_error_mm3"].abs().mean(),
    "median_abs_volume_error_mm3": df["volume_error_mm3"].abs().median(),
}

for k, v in summary.items():
    print(f"{k}: {v}")


In [ ]:
# Save CSV results
out_dir = Path("/content/drive/MyDrive/OpenPlaque/Validation")
out_dir.mkdir(parents=True, exist_ok=True)

csv_path = out_dir / "sample_dataset_validation.csv"
df.to_csv(csv_path, index=False)

summary_path = out_dir / "sample_dataset_validation_summary.txt"
with open(summary_path, "w") as f:
    for k, v in summary.items():
        f.write(f"{k}: {v}\n")

print("Saved:", csv_path)
print("Saved:", summary_path)


## Show best and worst cases by Dice

In [ ]:
if len(df) > 0:
    best_case = df.sort_values("dice", ascending=False).iloc[0]
    worst_case = df.sort_values("dice", ascending=True).iloc[0]

    print("Best case:", best_case["case_id"], "Dice:", best_case["dice"])
    print("Worst case:", worst_case["case_id"], "Dice:", worst_case["dice"])


In [ ]:
def show_case(row, label=2):
    img, vol = load_array(row["image"])
    _, gt = load_array(row["label"])
    _, pred = load_array(row["prediction"])

    counts = np.sum((gt == label) | (pred == label), axis=(1, 2))
    z = int(np.argmax(counts))

    plt.figure(figsize=(15, 5))

    plt.subplot(1, 3, 1)
    plt.imshow(vol[z], cmap="gray", vmin=-200, vmax=800)
    plt.title(f"{row['case_id']} image z={z}")
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.imshow(vol[z], cmap="gray", vmin=-200, vmax=800)
    plt.imshow(gt[z] == label, alpha=0.6)
    plt.title("Ground truth plaque")
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.imshow(vol[z], cmap="gray", vmin=-200, vmax=800)
    plt.imshow(pred[z] == label, alpha=0.6)
    plt.title("Predicted plaque")
    plt.axis("off")

    plt.show()


if len(df) > 0:
    show_case(best_case)
    show_case(worst_case)


## Optional: validate both label 1 and label 2

The primary TPV analysis uses label 2 = plaque. If desired, this cell summarizes both label classes.


In [ ]:
if len(df) > 0:
    both_rows = []
    for _, row in df.iterrows():
        _, pred = load_array(row["prediction"])
        _, gt = load_array(row["label"])

        for label in [1, 2]:
            m = binary_metrics(pred, gt, label=label)
            both_rows.append({
                "case_id": row["case_id"],
                "label": label,
                **m,
            })

    df_both = pd.DataFrame(both_rows)
    display(df_both.groupby("label")[["dice", "iou", "precision", "recall", "specificity"]].mean())

    both_csv = out_dir / "sample_dataset_validation_labels_1_2.csv"
    df_both.to_csv(both_csv, index=False)
    print("Saved:", both_csv)
